# K-means groups and score trajectories

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Plot unnormalized score trajectories by diagnosis and cluster, display diagnosis composition of the two clusters, and export pairwise group comparisons at each time point. This notebook uses existing cluster assignments; it does not fit k-means.


## Inputs

- `files/interm_files/full_subscales_trajectories_withoutnorm_{coh}.xlsx`: raw trajectories with ID, diagnosis codes, and score columns.
- `files/interm_files/kmeans{coh}.xlsx`: existing ID/CODE/IDX assignments, with IDX coded 0/1.

The default is `coh = [1, 2]` - discovery cohort, change to `coh = [3]` for valication cohort.

## Outputs

Under `files/plots/`, each figure is saved as PNG, SVG, TIFF, and PDF:

- `raw_{cohort}.<format>`: raw trajectories by diagnosis.
- `{cohort}_kmeanspie.<format>`: cluster composition.
- `clust_allmeasures_trajectory_{cohort}.<format>`: trajectories by diagnosis and cluster.

Under `files/tables/`:

- `NEW{cohort}_raw_kruskal_posthoc_results.xlsx`: raw-trajectory comparisons in long format.
- `{cohort}_raw_kruskal_posthoc_results.xlsx`: compact raw-trajectory comparisons.
- `{cohort}_clust_kruskal_posthoc_results.xlsx`: clustered comparisons; the final long table overwrites the compact table written earlier to the same path.





## 1. Setup and plotting style

Configure portable paths, typography, and diagnosis colors. Figure settings and export resolution are retained.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import kruskal
from matplotlib.lines import Line2D
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import matplotlib.gridspec as gridspec

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def set_global_plot_style():

    sns.set_context("paper", font_scale=1)

    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=8)
    plt.rc('axes', titlesize=8)
    plt.rc('axes', labelsize=8)
    plt.rc('xtick', labelsize=8)
    plt.rc('ytick', labelsize=8)
    plt.rc('legend', fontsize=8)
    plt.rc('figure', titlesize=8)

set_global_plot_style()

np.set_printoptions(precision=3, suppress=True)
import matplotlib.pyplot as plt
print(plt.rcParams['font.size'])
print(plt.rcParams['axes.titlesize'])
print(plt.rcParams['axes.labelsize'])


In [ ]:
colors = {
    ('HC', 0): '#0072B2',
    ('HC', 1): '#56B4E9',
    ('HC', 2): '#003F5C',
    ('AD', 0): '#D55E00',
    ('AD', 1): '#B35806',
    ('AD', 2): '#E69F00',
    ('PPD', 0): '#009E73',
    ('PPD', 1): '#66C2A5',
    ('PPD', 2): '#00441B',
}

FIGSIZE_2X1 = (14, 6)
FIGSIZE_2X2 = (14, 12)
FIGSIZE_1X1 = (7, 6)
figsize_new = ()

custom_palette = {
    'HC': colors[('HC', 0)],
    'AD': colors[('AD', 0)],
    'PPD': colors[('PPD', 0)]
}


## 2. Load raw trajectories

Select the cohort and rename ND → HC and AS → AD for display. Reshape scores to long format. MPAS T0 is set to missing because no baseline MPAS measurement is available in this workflow.

In [ ]:
coh = [1, 2]
input_raw_disc = OUTPUT_FILES / f'full_subscales_trajectories_withoutnorm_{coh}.xlsx'
raw_disc = pd.read_excel(input_raw_disc)
raw_disc


In [ ]:
print(type(raw_disc))
raw_disc.rename(columns={'CODE': 'Diagnosis'}, inplace=True)
raw_disc['Diagnosis'].replace({'ND': 'HC', 'AS': 'AD'}, inplace=True)
# Placeholder only: MPAS was not measured at T0.
raw_disc["MPAS T0"] = np.nan

mood_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Mood T0', 'Mood T1', 'Mood T2', 'Mood T3', 'Mood T4'],
    var_name='Time',
    value_name='Mood')

stress_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Stress T0', 'Stress T1', 'Stress T2', 'Stress T3', 'Stress T4'],
    var_name='Time',
    value_name='Stress'
)
mpas_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['MPAS T0', 'MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4'],
    var_name='Time',
    value_name='MPAS'
)

anhedonia_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Anhedonie_t0', 'Anhedonie_t1', 'Anhedonie_t2', 'Anhedonie_t3', 'Anhedonie_t4'],
    var_name='Time',
    value_name='Anhedonia'
)

anxiety_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Anxiety_t0', 'Anxiety_t1', 'Anxiety_t2', 'Anxiety_t3', 'Anxiety_t4'],
    var_name='Time',
    value_name='Anxiety'
)

depression_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Depression_t0', 'Depression_t1', 'Depression_t2', 'Depression_t3', 'Depression_t4'],
    var_name='Time',
    value_name='Depression'
)

epds_df = raw_disc.melt(
        id_vars=['ID', 'Diagnosis'],
        value_vars=['EPDStotal_t0_y', 'EPDStotal_t1_y', 'EPDStotal_t2_y', 'EPDStotal_t3_y', 'EPDStotal_t4_y'],
        var_name='Time',
        value_name='EPDS'
    )
epds_df


## 3. Diagnosis comparisons and raw trajectory plots

Plot group means with Seaborn’s 95% bootstrap confidence bands; no bootstrap seed is specified. For each measure and time point, run separate two-group Kruskal–Wallis tests for HC–AD, HC–PPD, and AD–PPD, skipping unavailable groups. Apply Benjamini–Hochberg correction only across the available pairs within that measure/time point, not across all measures or time points. Stars indicate adjusted significance at 0.05. These are time-specific comparisons, not a longitudinal repeated-measures model. Exported statistics and p-values are rounded to three decimals.

In [ ]:
dfs = {
    'Stress': stress_df,
    'Mood': mood_df,
    'MPAS': mpas_df,
    'Anhedonia': anhedonia_df,
    'Anxiety': anxiety_df,
    'Depression': depression_df,
    'EPDS': epds_df
}

custom_palette = {
    'HC': colors[('HC', 0)],
    'AD': colors[('AD', 0)],
    'PPD': colors[('PPD', 0)]
}

timepoints = ["T0", "T1", "T2", "T3", "T4"]
timepoint_numeric = list(range(len(timepoints)))
diagnosis_pairs = [('HC', 'AD'), ('HC', 'PPD'), ('AD', 'PPD')]
symptoms = ['Stress', 'Mood', 'MPAS', 'Anhedonia', 'Anxiety', 'Depression', 'EPDS']


In [ ]:
def plot_combined_4x2(symptom_list, fig_name, figsize=(7, 12)):
    """Plot diagnosis means and export time-specific pairwise comparisons."""
    results_list = []
    n_plots = len(symptom_list)
    n_rows = 4
    n_cols = 2

    fig = plt.figure(figsize=figsize, dpi=600)
    gs = gridspec.GridSpec(nrows=n_rows*2, ncols=n_cols, height_ratios=[3, 1.2]*n_rows, hspace=0.7, wspace=0.5)

    for idx, symptom in enumerate(symptom_list):
        print(symptom)
        row = (idx // n_cols) * 2
        col = idx % n_cols

        df = dfs[symptom].copy()

        ax_main = fig.add_subplot(gs[row, col])
        ax_sig = fig.add_subplot(gs[row + 1, col], sharex=ax_main)

        sns.lineplot(
            data=df,
            x='Time',
            y=symptom,
            hue='Diagnosis',
            style='Diagnosis',
            markers={
                'HC': 'o',
                'AD': 's',
                'PPD': '^'
            },
            dashes=False,
            palette=custom_palette,
            ax=ax_main,
            linewidth=1,
            markersize=4.5,
            errorbar=('ci', 95)
        )

        if symptom == 'Stress':
            title = 'Stress self-assessment'
        elif symptom == 'Mood':
            title = 'Mood self-assessment'
        elif symptom in ['Anhedonia', 'Anxiety', 'Depression']:
            title = f"{symptom} (EPDS)"
        else:
            title = symptom

        ax_main.set_title(title, fontsize=8)
        ax_main.set_xticks([])
        ax_main.set_xticklabels([])
        ax_main.set_xlabel("")
        ax_main.set_ylabel(f"Mean {symptom} Score", fontsize=8)
        ax_main.tick_params(labelsize=9)
        ax_main.grid(True, axis='y', color='lightgrey', linestyle='--', linewidth=0.75)
        ax_main.grid(False, axis='x')

        ax_main.get_legend().remove()

        if symptom in ['Stress', 'Mood']:
            ax_main.set_ylim(0, 10)
            ax_main.set_yticks(range(0, 11, 2))
        elif symptom == 'MPAS':
            ax_main.set_ylim(70, 95)
            ax_main.set_yticks(range(70, 96, 5))
        elif symptom == 'Anhedonia':
            ax_main.set_ylim(0, 6)
            ax_main.set_yticks(range(0, 7, 2))
        elif symptom == 'Anxiety':
            ax_main.set_ylim(0, 9)
            ax_main.set_yticks(range(0, 11, 2))
        elif symptom == 'Depression':
            ax_main.set_ylim(0, 12)
            ax_main.set_yticks(range(0, 13, 2))
        elif symptom == 'EPDS':
            ax_main.set_ylim(0, 18)
            ax_main.set_yticks(range(0, 19, 2))

        df['Time'] = df['Time'].str.extract(r'[_ ]?t?(\d)').iloc[:, 0]
        df['Time'] = 'T' + df['Time']

        star_matrix = np.full((len(diagnosis_pairs), len(timepoints)), '', dtype=object)

        for i, t in enumerate(timepoints):
            print(t)
            subset = df[df['Time'] == t]
            pvals = []
            stats = []
            tested_pairs = []

            for j, (d1, d2) in enumerate(diagnosis_pairs):
                print(d1,d2)
                pair_df = subset[subset['Diagnosis'].isin([d1, d2])]
                if pair_df['Diagnosis'].nunique() < 2:
                    continue
                group_scores = [grp[symptom].dropna().values for _, grp in pair_df.groupby('Diagnosis')]
                if any(len(scores) == 0 for scores in group_scores):
                    continue
                stat, p_kw = kruskal(*group_scores)
                stats.append(stat)
                pvals.append(p_kw)
                tested_pairs.append(j)
                print(pvals)

            if pvals:
                print('Significant')
                # Correct only the available diagnosis pairs at this measure/time point.
                reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
                print(pvals_corrected)

                for passed, idx_pair, stat, realp, fdr_p in zip(reject, tested_pairs, stats, pvals, pvals_corrected):
                    d1, d2 = diagnosis_pairs[idx_pair]
                    if passed:
                        star_matrix[idx_pair, i] = '*'
                    print(star_matrix)

                    results_list.append({
                    "Symptom": symptom,
                    "Timepoint": t,
                    "Comparison": f"{d1} vs {d2}",
                    "H": round(stat, 3),
                    "p_value": round(realp, 3),
                    "FDR_p_value": round(fdr_p, 3),
                })

                results_df = pd.DataFrame(results_list)

                if coh == [3]:
                    results_df.to_excel(OUTPUT_TABLES / "NEWval_raw_kruskal_posthoc_results.xlsx", index=False)
                elif coh == [1, 2]:
                    results_df.to_excel(OUTPUT_TABLES / "NEWdisc_raw_kruskal_posthoc_results.xlsx", index=False)

        for j, (d1, d2) in enumerate(diagnosis_pairs):
            for i in range(len(timepoints)):
                star = star_matrix[j, i]
                if star:

                    ax_sig.hlines(y=j, xmin=i - 0.2, xmax=i + 0.2, color='black', linewidth=1)

                    ax_sig.text(i, j, star, ha='center', va='center', fontsize=8, color='black')

        ax_sig.set_yticks(range(len(diagnosis_pairs)))
        ax_sig.set_yticklabels([f"{d1} vs {d2}" for d1, d2 in diagnosis_pairs],
                               fontsize=8, ha='right')

        ax_sig.set_xticks(timepoint_numeric)
        ax_sig.set_xticklabels(timepoints, fontsize=8)
        ax_sig.set_xlim(-0.5, len(timepoints) - 0.5)
        ax_sig.set_ylim(-0.5, len(diagnosis_pairs) - 0.5)
        ax_sig.invert_yaxis()

        if symptom in ['Depression', 'MPAS']:
            ax_sig.set_xlabel('Time point', fontsize=8)
        else:
            ax_sig.set_xlabel("")

        for spine in ax_sig.spines.values():
            spine.set_visible(False)
        ax_sig.tick_params(left=False, bottom=False)
        ax_sig.grid(False)

    if n_plots < n_rows * n_cols:
        row = (n_plots // n_cols) * 2
        col = n_plots % n_cols
        ax_empty_main = fig.add_subplot(gs[row, col])
        ax_empty_sig = fig.add_subplot(gs[row + 1, col])
        ax_empty_main.axis('off')
        ax_empty_sig.axis('off')

        handles, labels = ax_main.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            title='Diagnosis',
            loc='upper center',
            ncol=len(labels),
            bbox_to_anchor=(0.5, 0.95),
            frameon=False,
            fontsize=8,
            title_fontsize=8
        )

    plt.tight_layout()

    for ext in ['png', 'svg', 'tiff', 'pdf']:
        save_path = OUTPUT_PLOTS / f"{fig_name}.{ext}"
        fig.savefig(save_path, dpi=800, bbox_inches='tight', format=ext)

    plt.show()

    results_df = results_df.sort_values(by=["Symptom", "Timepoint", "Comparison"])

    grouped = results_df.groupby(["Symptom", "Comparison"])

    compact_df = grouped.agg({
        "Timepoint": lambda x: ', '.join(x),
        "H": lambda x: ', '.join(map(str, x)),
        "p_value": lambda x: ', '.join(map(str, x)),
        "FDR_p_value": lambda x: ', '.join(map(str, x))
    }).reset_index()

    if coh == [3]:
        compact_df.to_excel(OUTPUT_TABLES / "val_raw_kruskal_posthoc_results.xlsx", index=False)
    elif coh == [1, 2]:
        compact_df.to_excel(OUTPUT_TABLES / "disc_raw_kruskal_posthoc_results.xlsx", index=False)

if coh == [3]:
    suf = 'val'
else:
    suf = 'disc'

symptoms_ordered = ['Stress', 'Mood', 'EPDS', 'Anhedonia', 'Anxiety', 'Depression', 'MPAS']
plot_combined_4x2(symptoms_ordered, fig_name=f'raw_{suf}', figsize=(7, 10))


### Standalone FDR calculation

The following cell adjusts four manually supplied p-values. It is independent of the plotted comparisons and does not change their results.

In [ ]:
pvalsnew = [2.192220734051422e-08, 0.1546085061750441, 0.9452157140888655, 0.21871981683079905]

reject, pvals_corrected, _, _ = multipletests(pvalsnew, alpha=0.05, method='fdr_bh')
reject, pvals_corrected


In [ ]:
coh


## 4. Plot cluster composition

Read existing assignments and relabel clusters 0/1 as 1/2 for presentation. Donut labels report both the diagnosis percentage within a cluster and the percentage of that diagnosis assigned to the cluster. Run the relabeling step once per fresh load.

In [ ]:
input_clust_disc = OUTPUT_FILES / f'kmeans{coh}.xlsx'
clust_disc = pd.read_excel(input_clust_disc)

clust_disc = clust_disc.rename(columns={'Diagnosis': 'CODE'})
clust_disc = clust_disc.rename(columns={'Cluster': 'IDX'})
# Convert stored 0/1 labels to displayed clusters 1/2.
clust_disc['IDX'] = clust_disc['IDX'].replace({0: 1, 1:2})
clust_disc.columns


In [ ]:
coh


In [ ]:
clust_disc.groupby('IDX')['CODE'].value_counts()


In [ ]:
from matplotlib.patches import Patch

diagnosis_labels = {'HC': 'HC', 'AD': "AD", 'PPD': 'PPD'}
colors = {
    ('HC', 0): '#0072B2',
    ('AD', 0): '#D55E00',
    ('PPD', 0): '#009E73',
    ('HC', 1): '#0072B2',
    ('AD', 1): '#D55E00',
    ('PPD', 1): '#009E73',
    ('HC', 2): '#0072B2',
    ('AD', 2): '#D55E00',
    ('PPD', 2): '#009E73'
}

clust_disc['CODE'] = clust_disc['CODE'].replace({'ND': 'HC', 'AS': 'AD'})
group_counts = clust_disc.groupby(['CODE', 'IDX']).size().reset_index(name='Count')

cluster1_data = group_counts[group_counts['IDX'] == 1].copy()
cluster2_data = group_counts[group_counts['IDX'] == 2].copy()

cluster1_total = cluster1_data['Count'].sum()
cluster2_total = cluster2_data['Count'].sum()
total = cluster1_total + cluster2_total

overall_totals = clust_disc['CODE'].value_counts().to_dict()

fig, axs = plt.subplots(1, 2, figsize=(7, 5), dpi=800, gridspec_kw={'wspace': 0.05})
fig.subplots_adjust(top=0.8)

def plot_donut(ax, cluster_data, cluster_id, cluster_total, overall_totals):
    """Draw one cluster with within-cluster and within-diagnosis percentages."""
    sizes = []
    labels = []
    color_list = []

    for _, row in cluster_data.iterrows():
        diag = row['CODE']
        count = row['Count']
        # Distinguish cluster composition from the distribution of each diagnosis.
        percent_in_cluster = (count / cluster_total) * 100
        total_diag = overall_totals.get(diag, 1)
        percent_of_diag = (count / total_diag) * 100

        label = f"{percent_in_cluster:.1f}% of Cluster {cluster_id}\n({percent_of_diag:.1f}% of all {diag})"
        sizes.append(count)
        labels.append(label)
        color_list.append(colors.get((diag, cluster_id), '#cccccc'))

    wedges, _ = ax.pie(
        sizes,
        labels=None,
        colors=color_list,
        startangle=90,
        wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5, alpha=0.95)
    )

    for i, wedge in enumerate(wedges):
        angle = (wedge.theta2 + wedge.theta1) / 2
        angle_rad = np.deg2rad(angle)
        x = np.cos(angle_rad)
        y = np.sin(angle_rad)
        ha = 'left' if x > 0 else 'right'

        if cluster_id == 2 and cluster_data.iloc[i]['CODE'] == 'AD':
            y_offset = 0.50
        else:
            y_offset = 0

        ax.annotate(
            labels[i],
            xy=(x * 0.7, y * 0.7),
            xytext=(x * 1, y * 1.15 + y_offset),
            ha=ha,
            va='center',
            fontsize=8,
            arrowprops=dict(arrowstyle='-', connectionstyle='angle', color='black')
        )

    ax.text(0, 0, f"Cluster {cluster_id}\nn={cluster_total}", ha='center', va='center', color='#333333', fontsize=8)
    ax.set(aspect="equal")

plot_donut(axs[0], cluster1_data, cluster_id=1, cluster_total=cluster1_total, overall_totals=overall_totals)
plot_donut(axs[1], cluster2_data, cluster_id=2, cluster_total=cluster2_total, overall_totals=overall_totals)

if coh == [3]:
    titleis = 'Validation'
    aux_name = 'val'
elif coh == [1, 2]:
    titleis = 'Discovery'
    aux_name = 'disc'

fig.suptitle(f"{titleis} Cohort n={total}", y=0.75, fontsize=8)

if coh == [1, 2]:
    legend_elements = [
        Patch(facecolor=colors[('HC', 0)], edgecolor='white', label='HC'),
        Patch(facecolor=colors[('AD', 0)], edgecolor='white', label='AD'),
        Patch(facecolor=colors[('PPD', 0)], edgecolor='white', label='PPD'),
    ]
    fig.legend(
        handles=legend_elements,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.85),
        ncol=3,
        frameon=False,
        fontsize=9,
        title_fontsize=9,
        title='Diagnosis'
    )

for ext in ['tiff', 'svg', 'png', 'pdf']:
    fig.savefig(OUTPUT_PLOTS / f"{aux_name}_kmeanspie.{ext}", dpi=800, bbox_inches='tight', format=ext)

plt.show()


In [ ]:
raw_disc['Diagnosis'].value_counts()
raw_disc.columns


## 5. Join raw scores to cluster assignments

Inner-join on ID and diagnosis code, retaining matching records, and export the joined table. Participant IDs are assumed unique. Reshape the joined scores into diagnosis-by-cluster trajectories.

In [ ]:
if coh == [1, 2]:
    raw_disc['CODE'] = raw_disc['CODE_x']
    raw_disc['CODE'] = raw_disc['CODE'].replace({'ND': 'HC', 'AS': 'AD'})
    merged = pd.merge(raw_disc, clust_disc[['ID', 'CODE', 'IDX']], on=['ID', 'CODE'], how='inner')
elif coh == [3]:
    raw_disc['CODE'] = raw_disc['Diagnosis']
    merged = pd.merge(raw_disc, clust_disc[['ID', 'CODE', 'IDX']], on=['ID', 'CODE'], how='inner')

merged.columns
if coh == [3]:
    merged.to_excel(OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx", index=False)
elif coh == [1, 2]:
    merged.to_excel(OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx", index=False)


In [ ]:
merged


In [ ]:
long_list = []

if coh == [1, 2]:
    symptom_columns = {
        'Stress': [f'Stress T{i}' for i in range(5)],
        'Mood': [f'Mood T{i}' for i in range(5)],
        'MPAS': [f'MPAS T{i}' for i in range(5)],
        'Anhedonia': [f'Anhedonie_t{i}' for i in range(5)],
        'Anxiety': [f'Anxiety_t{i}' for i in range(5)],
        'Depression': [f'Depression_t{i}' for i in range(5)],
        'EPDS': [f'EPDStotal_t{i}_y' for i in range(5)]
    }
elif coh == [3]:
        symptom_columns = {
        'Stress': [f'Stress T{i}' for i in range(5)],
        'Mood': [f'Mood T{i}' for i in range(5)],
        'MPAS': [f'MPAS T{i}' for i in range(5)],
        'Anhedonia': [f'Anhedonie_t{i}' for i in range(5)],
        'Anxiety': [f'Anxiety_t{i}' for i in range(5)],
        'Depression': [f'Depression_t{i}' for i in range(5)],
        'EPDS': [f'EPDStotal_t{i}_y' for i in range(5)]
    }

for symptom, cols in symptom_columns.items():
    df_long = merged.melt(
        id_vars=['ID', 'CODE', 'IDX'],
        value_vars=cols,
        var_name='Time',
        value_name='Score'
    )
    df_long['Symptom'] = symptom
    df_long['Time'] = df_long['Time'].str.extract(r'[_ ]?t?(\d)').iloc[:, 0]
    df_long['Time'] = 'T' + df_long['Time']
    long_list.append(df_long)

df_long = pd.concat(long_list, ignore_index=True)
df_long['Group'] = df_long.apply(lambda row: f"{row['CODE']} (C{row['IDX']})", axis=1)

group_palette = {'AD (C2)': '#B35806',
 'AD (C1)': '#E69F00',
 'HC (C2)': '#003F5C',
 'HC (C1)': '#56B4E9',
 'PPD (C2)': '#007a59'}

linestyles = {
    'HC (C1)': (1, 0),
    'HC (C2)': (4, 2),

    'AD (C1)': (1, 0),
    'AD (C2)': (4, 2),

    'PPD (C2)': (4, 2)
}
group_palette
df_long


In [ ]:
colors


## 6. Plot clustered trajectories and group comparisons

Plot means with 95% bootstrap confidence bands. At each measure/time point, compare HC C1–HC C2, HC C2–AD C2, AD C2–PPD C2, and HC C2–PPD C2 using two-group Kruskal–Wallis tests. Apply BH correction across the available comparisons at that measure/time point. Missing groups are skipped. Stars use unrounded adjusted p-values; exports round to three decimals. The final export replaces the compact comparison table with the long-format results.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import seaborn as sns

group_pairs = [
    ('HC (C1)', 'HC (C2)'),
    ('HC (C2)', 'AD (C2)'),
    ('AD (C2)', 'PPD (C2)'),
    ('HC (C2)', 'PPD (C2)')
]
results_list = []

all_symptoms = ['Stress', 'Mood', 'EPDS', 'Anhedonia', 'Anxiety', 'Depression', 'MPAS']
fig = plt.figure(figsize=(7, 13), dpi=600)
gs = gridspec.GridSpec(nrows=8, ncols=2, height_ratios=[3, 1.2] * 4, hspace=0.6, wspace=0.5)

for idx, symptom in enumerate(all_symptoms):
    print(symptom)
    df_sym = df_long[df_long['Symptom'] == symptom]
    row = (idx // 2) * 2
    col = idx % 2

    ax_main = fig.add_subplot(gs[row, col])
    ax_sig = fig.add_subplot(gs[row + 1, col], sharex=ax_main)

    sns.lineplot(
        data=df_sym,
        x='Time',
        y='Score',
        hue='Group',
        style='Group',
        palette=group_palette,
        dashes=linestyles,
        markers={
            'HC (C1)': 'o',
            'HC (C2)': 'o',
            'AD (C1)': 's',
            'AD (C2)': 's',
            'PPD (C2)': '^'
        },
        markersize=4,
        linewidth=1,
        errorbar=('ci', 95),
        ax=ax_main,
        legend=False
    )
    ax_main.set_xlabel('')

    if symptom == 'Stress':
        title = 'Stress self-assessment'
    elif symptom == 'Mood':
        title = 'Mood self-assessment'
    elif symptom in ['Anhedonia', 'Anxiety', 'Depression']:
        title = f"{symptom} (EPDS)"
    else:
        title = symptom

    ax_main.set_title(title, fontsize=9)
    ax_main.set_ylabel(f"Mean {symptom} Score", fontsize=9)
    ax_main.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

    ax_main.set_xticks([])
    ax_main.set_xticklabels([])
    ax_main.tick_params(labelsize=9)
    ax_main.grid(True, axis='y', color='lightgrey', linestyle='--', linewidth=0.75)
    ax_main.grid(False, axis='x')

    if symptom in ['Stress', 'Mood']:
        ax_main.set_ylim(0, 10)
        ax_main.set_yticks(range(0, 11, 2))
    elif symptom == 'MPAS':
        ax_main.set_ylim(70, 95)
        ax_main.set_yticks(range(70, 96, 5))
    elif symptom == 'Anhedonia':
        ax_main.set_ylim(0, 6)
        ax_main.set_yticks(range(0, 7, 2))
    elif symptom == 'Anxiety':
        ax_main.set_ylim(0, 9)
        ax_main.set_yticks(range(0, 10, 2))
    elif symptom == 'Depression':
        ax_main.set_ylim(0, 12)
        ax_main.set_yticks(range(0, 13, 2))
    elif symptom == 'EPDS':
        ax_main.set_ylim(0, 18)
        ax_main.set_yticks(range(0, 19, 2))

    star_matrix = np.full((len(group_pairs), len(timepoints)), '', dtype=object)

    for i, t in enumerate(timepoints):
        print(t)
        df_t = df_sym[df_sym['Time'] == t]
        pvals = []
        tested_pairs = []
        stats = []

        for j, (g1, g2) in enumerate(group_pairs):
            pair_df = df_t[df_t['Group'].isin([g1, g2])]
            if pair_df['Group'].nunique() < 2:
                continue
            group_scores = [grp['Score'].dropna().values for _, grp in pair_df.groupby('Group')]
            if any(len(scores) == 0 for scores in group_scores):
                continue
            stat, p_kw = kruskal(*group_scores)
            stats.append(stat)
            pvals.append(p_kw)
            tested_pairs.append(j)
            print(pvals)

        if pvals:
            # Correct the selected group comparisons within this measure/time point.
            reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
            print(pvals_corrected)
            for passed, idx_pair, stat, rp, fdr_p in zip(reject, tested_pairs, stats, pvals, pvals_corrected):
                d1, d2 = group_pairs[idx_pair]
                if passed:
                    star_matrix[idx_pair, i] = '*'
                results_list.append({
                    "Symptom": symptom,
                    "Timepoint": t,
                    "Comparison": f'{d1} vs {d2}',
                    "H": round(stat, 3),
                    'p-value': round(rp, 3),
                    "FDR p-value": round(fdr_p, 3)
                })

    results_df = pd.DataFrame(results_list)
    results_df = results_df.sort_values(by=["Symptom", "Timepoint", "Comparison"])

    grouped = results_df.groupby(["Symptom", "Comparison"])

    compact_df = grouped.agg({
        "Timepoint": lambda x: ', '.join(x),
        "H": lambda x: ', '.join(map(str, x)),
        "p-value": lambda x: ', '.join(map(str, x)),
        "FDR p-value": lambda x: ', '.join(map(str, x))
    }).reset_index()

    if coh == [3]:
        compact_df.to_excel(OUTPUT_TABLES / "val_clust_kruskal_posthoc_results.xlsx", index=False)
    elif coh == [1, 2]:
        compact_df.to_excel(OUTPUT_TABLES / "disc_clust_kruskal_posthoc_results.xlsx", index=False)
    for j, (g1, g2) in enumerate(group_pairs):
        y = j

        for t_idx in range(len(timepoints)):
            star = star_matrix[j, t_idx]
            if star:
                ax_sig.hlines(y=y, xmin=t_idx - 0.2, xmax=t_idx + 0.2, color='black', linewidth=1.5, zorder=1)
                ax_sig.text(t_idx, y, star, ha='center', va='center', fontsize=12, zorder=2)

    ax_sig.set_yticks(range(len(group_pairs)))
    ax_sig.set_yticklabels([f"{g1} vs {g2}" for g1, g2 in group_pairs], fontsize=8)
    ax_sig.set_xticks(timepoint_numeric)
    ax_sig.set_xticklabels(timepoints)
    ax_sig.set_xlim(-0.5, len(timepoints) - 0.5)
    ax_sig.set_ylim(-0.5, len(group_pairs) - 0.5)
    ax_sig.invert_yaxis()
    if symptom in ['MPAS', 'Depression']:
        ax_sig.set_xlabel('Time point', fontsize=8)
    else:
        ax_sig.set_xlabel('')

    for spine in ax_sig.spines.values():
        spine.set_visible(False)
    ax_sig.tick_params(left=False, bottom=False)
    ax_sig.grid(False)

markers_map = {
    'HC (C1)': 'o',
    'HC (C2)': 'o',
    'AD (C1)': 's',
    'AD (C2)': 's',
    'PPD (C2)': '^'
}

handles = []

for group in sorted(group_palette.keys()):
    h = Line2D(
        [0], [0],
        color=group_palette[group],
        linewidth=2,
        linestyle='-',
        marker=markers_map[group],
        markersize=5,
        markerfacecolor=group_palette[group],
        markeredgecolor='black',
        label=group
    )

    if linestyles[group]:
        h.set_dashes(linestyles[group])

    handles.append(h)

fig.legend(
    handles=handles,
    title='Group',
    loc='upper center',
    bbox_to_anchor=(0.5, 0.94),
    ncol=len(handles),
    frameon=False,
    fontsize=8,
    title_fontsize=8
)

if coh == [1, 2]:
    aux_name = 'disc'
elif coh == [3]:
    aux_name = 'val'

for ext in ['png', 'svg', 'tiff', 'pdf']:
    save_path = OUTPUT_PLOTS / f"clust_allmeasures_trajectory_{aux_name}.{ext}"
    fig.savefig(save_path, dpi=800, bbox_inches='tight', format=ext)

plt.show()

results_df = pd.DataFrame(results_list)
# This long-format export overwrites the compact table at the same path.
results_df.to_excel(OUTPUT_TABLES / f"{aux_name}_clust_kruskal_posthoc_results.xlsx", index=False)


In [ ]:
compact_df
